# Normalizacion de Texto con Programacion Funcional

Construimos un pipeline modular de limpieza usando `map`, `filter`, `reduce`
y composicion de funciones. Cada paso es una funcion pura reutilizable.

## Ejercicio 1: Funciones puras + `map`

**Concepto:** Funcion pura = mismo input, siempre mismo output, sin efectos secundarios.

1. Define funciones puras para: minusculas, quitar acentos, quitar especiales, colapsar espacios.
2. Aplica cada una con `map` sobre una lista de textos.
3. Observa como cada transformacion es independiente.

In [ ]:
import re, unicodedata
from functools import reduce

textos = [
    "  El   Procesamiento de  Lenguaje Natural (PLN) es fascinante!  ",
    "Gensim es una libreria de Python para modelado de topicos.",
    "Los embeddings capturan el SIGNIFICADO semantico del lenguaje.",
    "Word2Vec y FastText: algoritmos populares para crear embeddings.",
    "  La inteligencia artificial transforma como procesamos informacion.   ",
]

# Funciones puras (una responsabilidad cada una)
def a_minusculas(s):
    return s.lower()

def quitar_acentos(s):
    nfd = unicodedata.normalize('NFD', s)
    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')

def quitar_especiales(s):
    return re.sub(r'[^a-z\s]', '', s)

def colapsar_espacios(s):
    return re.sub(r'\s+', ' ', s).strip()

# Aplicar cada transformacion con map y ver el efecto
etapas = [
    ("Minusculas",     a_minusculas),
    ("Sin acentos",    quitar_acentos),
    ("Sin especiales", quitar_especiales),
    ("Espacios norm.", colapsar_espacios),
]

ejemplo = textos[0]
print(f"Original: {ejemplo.strip()}")
actual = ejemplo
for nombre, f in etapas:
    actual = f(actual)
    print(f"  -> {nombre}: {actual}")


## Ejercicio 2: Composicion de funciones

**Concepto:** `componer([f, g, h])(x)` equivale a `h(g(f(x)))`.
Con `reduce` podemos encadenar cualquier numero de pasos.

1. Implementa `componer` con `reduce`.
2. Construye dos pipelines distintos (basico y completo).
3. Aplica ambos al corpus con `map`.

In [ ]:
def componer(funciones):
    """Compone una lista de funciones en una sola (de izq a derecha)."""
    return reduce(lambda f, g: lambda x: g(f(x)), funciones)

# Pipelines declarativos: solo listas de pasos
pipeline_basico   = componer([a_minusculas, quitar_especiales, colapsar_espacios])
pipeline_completo = componer([a_minusculas, quitar_acentos, quitar_especiales, colapsar_espacios])

print("Comparacion de pipelines en el mismo texto:")
for t in textos[:3]:
    b = pipeline_basico(t)
    c = pipeline_completo(t)
    print(f"  Original : {t.strip()[:55]}")
    print(f"  Basico   : {b}")
    print(f"  Completo : {c}")
    print()

# Aplicar el pipeline completo a todo el corpus
corpus_limpio = list(map(pipeline_completo, textos))
print("Corpus normalizado:")
for r in corpus_limpio:
    print(f"  {r}")


## Ejercicio 3: `filter` con predicados componibles

**Concepto:** Un predicado es una funcion que devuelve `True` o `False`.
Podemos combinar predicados con una funcion de orden superior.

1. Define predicados para filtrar tokens.
2. Crea `todos_cumplen` que combina predicados con AND.
3. Aplica `filter` al corpus tokenizado.

In [ ]:
import nltk
from nltk.corpus import stopwords

for ruta, nombre in [('corpora/stopwords','stopwords'),
                      ('tokenizers/punkt_tab','punkt_tab')]:
    try: nltk.data.find(ruta)
    except LookupError: nltk.download(nombre, quiet=True)

STOPWORDS = set(stopwords.words('spanish'))

# Tokenizar con map
tokens_por_oracion = list(map(str.split, corpus_limpio))
todos = reduce(lambda a, b: a + b, tokens_por_oracion, [])
print(f"Total tokens: {len(todos)}")

# Predicados nombrados
def no_es_stopword(t):  return t not in STOPWORDS
def longitud_ok(t):     return len(t) > 2
def no_es_numero(t):    return not t.isdigit()

# Funcion de orden superior: AND de multiples predicados
def todos_cumplen(predicados):
    """Devuelve un predicado que es True solo si todos los predicados lo son."""
    return lambda x: all(pred(x) for pred in predicados)

predicado_final = todos_cumplen([no_es_stopword, longitud_ok, no_es_numero])
tokens_validos  = list(filter(predicado_final, todos))

eliminados = len(todos) - len(tokens_validos)
print(f"Eliminados : {eliminados} ({eliminados/len(todos)*100:.1f}%)")
print(f"Validos    : {len(tokens_validos)}")
print(f"Primeros 15: {tokens_validos[:15]}")


## Ejercicio 4: Frecuencias y reporte con `reduce` y `map`

1. Construye el diccionario de frecuencias usando solo `reduce` (sin Counter).
2. Usa `map` para formatear el reporte final.
3. Encuentra minimo, maximo y promedio de longitudes con funciones puras.

In [ ]:
# Frecuencias con reduce
def contar(acum, token):
    acum[token] = acum.get(token, 0) + 1
    return acum

frecuencias = reduce(contar, tokens_validos, {})
top10 = sorted(frecuencias.items(), key=lambda par: par[1], reverse=True)[:10]

# Reporte formateado con map
formatear = lambda par: f"  {par[0]:<22} {par[1]:>4}  {'#' * par[1]}"
lineas = list(map(formatear, top10))

print("Top 10 palabras:")
for l in lineas:
    print(l)

# Estadisticas de longitud con map y reduce
longitudes  = list(map(len, tokens_validos))
total_chars = reduce(lambda a, b: a + b, longitudes)
maximo      = reduce(lambda a, b: a if a >= b else b, longitudes)
minimo      = reduce(lambda a, b: a if a <= b else b, longitudes)

print(f"\nLongitud minima : {minimo}")
print(f"Longitud maxima : {maximo}")
print(f"Longitud promedio: {total_chars/len(longitudes):.2f}")

# Palabra mas frecuente con reduce
mas_freq = reduce(lambda a, b: a if a[1] >= b[1] else b, frecuencias.items())
print(f"Palabra mas frecuente: '{mas_freq[0]}' ({mas_freq[1]} veces)")
